# 信噪 · GRPO 后训练（Kaggle 瘦壳）

在 SFT LoRA adapter 之上做 GRPO，使用混合 reward（LLM-as-judge 主体 + 规则项）。

## 准备
1. 把代码 + `data/grpo_train.jsonl` 打包成 Kaggle Dataset 加进来。
   `grpo_train.jsonl` 必须用**当前版本**的 `python scripts/build_grpo_dataset.py`
   重新生成 —— 现在每行是对话式 prompt（带 SFT 同款 system 人设），旧的裸字符串
   数据会让 GRPO 在和 warm-start / 评测都不一致的分布上训歪。
2. 把 SFT 训出来的 `xinzao-lora-adapter` Dataset 加进来
3. **Add-ons → Secrets**: `DEEPSEEK_API_KEY`（reward function 调 judge API 用）
4. Settings: GPU T4 x2 / P100, Internet On

**预估**: 65 prompts × 4 generations × 2 epochs ≈ 520 次 judge 调用，约 20-40 分钟，DeepSeek API 成本 < \$1。

In [ ]:
# 1. 安装依赖
!pip uninstall -y -q transformers trl datasets unsloth unsloth-zoo
!pip install -q unsloth bitsandbytes
!pip install -q peft accelerate  # --no-unsloth 路径需要 peft
!pip install -q openai python-dotenv  # judge 调 DeepSeek 要用
# trl 这版对 mergekit / llm-blender 做了硬 import（非可选），必须装
!pip install -q mergekit llm-blender

In [ ]:
# 2. 注入 Secrets + 找路径
import os, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["DEEPSEEK_API_KEY"] = secrets.get_secret("DEEPSEEK_API_KEY")

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/repo")

# 找代码目录
script_hits = list(INPUT_ROOT.rglob("scripts/train_grpo.py"))
if script_hits:
    code_root = script_hits[0].parents[1]
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    shutil.copytree(code_root, WORK_DIR)
else:
    !git clone https://github.com/alchosyn/npc-dialogue-ai-agent.git $WORK_DIR
os.chdir(WORK_DIR)

# 找训练数据（grpo_train.jsonl）
data_hits = list(INPUT_ROOT.rglob("grpo_train.jsonl"))
if data_hits:
    TRAIN_JSONL = data_hits[0]
else:
    # 回退：本地有 cases.json + sft_seeds.json 就现场生成
    print("未找到 grpo_train.jsonl，本地生成中...")
    !python scripts/build_grpo_dataset.py
    TRAIN_JSONL = WORK_DIR / "data" / "grpo_train.jsonl"

# 找 SFT adapter
all_hits = list(INPUT_ROOT.rglob("adapter_config.json"))
final_hits = [p for p in all_hits if "checkpoint-" not in str(p)]
adapter_hits = final_hits if final_hits else all_hits
assert adapter_hits, "找不到 SFT LoRA adapter"
SFT_ADAPTER = adapter_hits[0].parent

OUTPUT_DIR = Path("/kaggle/working/qwen-1.5b-xinzao-grpo")
print(f"工作目录: {WORK_DIR}")
print(f"训练数据: {TRAIN_JSONL}")
print(f"SFT adapter: {SFT_ADAPTER}")
print(f"输出: {OUTPUT_DIR}")

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} (bf16: {torch.cuda.is_bf16_supported()})")

In [ ]:
# 3. 跑 GRPO 训练
import subprocess, sys
cmd = [
    sys.executable, "scripts/train_grpo.py",
    "--train-jsonl", str(TRAIN_JSONL),
    "--sft-adapter", str(SFT_ADAPTER),
    "--output-dir", str(OUTPUT_DIR),
    "--no-unsloth",  # Kaggle 上 Unsloth 与 transformers 有 apply_qkv 兼容 bug，用原生加载
    # batch/num_generations 必须自洽：batch 与 batch×grad-accum 都要能被
    # num_generations 整除。4/4/4 是 16GB T4 上安全自洽的一组（脚本默认值也是这组）。
    "--num-generations", "4",
    "--batch-size", "4",
    "--grad-accum", "4",
    "--max-prompt-length", "384",
    "--max-completion-length", "384",
    "--num-train-epochs", "2",
    "--learning-rate", "1e-6",
    "--judge-workers", "8",
    "--precision", "auto",
]
print("+ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 4. 看输出
!ls -lh /kaggle/working/qwen-1.5b-xinzao-grpo
print("\n下一步：把 qwen-1.5b-xinzao-grpo 转 Dataset，然后开 eval notebook 跑 5 路对比")